# AlphaFold Ensemble Competition - Tournament Edition

This notebook screens a list of potential peptide binders using a 'Winner Stays On' (King of the Hill) tournament structure.
It evaluates pairs of binders against the target protein. The winner advances to face the next candidate on the list.

In [ ]:
import os
import py3Dmol
import numpy as np
from af_competition import run_colabfold, process_ensemble, optimize_threshold  # noqa: F401


## 1. Configuration
Define the target, binding site, and list of potential binders.

In [ ]:
TARGET_SEQ = "SQIPASEQETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDAAQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVNQQ"
BINDING_SITE_RESIDUES = [42, 84]  # 1-indexed on Target Chain

# List of potential binders to screen in the tournament
BINDER_CANDIDATES = [
    "ETFSDLWKLLPE", # Candidate 0
    "LTFEHYWAQLTS", # Candidate 1
    "ARGF",         # Candidate 2 (Example)
]

NUM_SEEDS = 20
BASE_OUTPUT_DIRECTORY = "./colabfold_results"
MIN_PLDDT = 80.0


## 2. Tournament Execution
Runs the 'Winner Stays On' tournament.

In [ ]:
champion_idx = 0
champion_seq = BINDER_CANDIDATES[0]

last_match_dir = ""
last_match_stats = {}
winning_state_last_match = ""

print(f"Starting Tournament with {len(BINDER_CANDIDATES)} candidates.\n")

for challenger_idx in range(1, len(BINDER_CANDIDATES)):
    challenger_seq = BINDER_CANDIDATES[challenger_idx]
    run_name = f"tournament_match_{champion_idx}_vs_{challenger_idx}"
    
    print(f"--- Match {challenger_idx}: Champion [{champion_idx}] vs Challenger [{challenger_idx}] ---")
    
    # --- 1. Execute ColabFold ---
    # Uncomment the line below to actively generate new models:
    # ACTUAL_OUTPUT_DIR = run_colabfold(TARGET_SEQ, champion_seq, challenger_seq, BASE_OUTPUT_DIRECTORY, run_name=run_name, num_seeds=NUM_SEEDS)
    
    # Fallback directory if running analysis on previously generated models:
    ACTUAL_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIRECTORY, run_name)
    last_match_dir = ACTUAL_OUTPUT_DIR
    
    if not os.path.exists(ACTUAL_OUTPUT_DIR):
        print(f"Directory {ACTUAL_OUTPUT_DIR} not found! Uncomment 'run_colabfold' above to generate models.\n")
        continue
        
    # --- 2. Analyze ---
    all_results = process_ensemble(ACTUAL_OUTPUT_DIR, BINDING_SITE_RESIDUES, distance_threshold=0.0)
    results = [r for r in all_results if r["mean_plddt"] >= MIN_PLDDT]
    
    if not results:
        print(f"Match void: No models passed pLDDT threshold {MIN_PLDDT}.")
        print(f"Champion [{champion_idx}] retains title by default.\n")
        winning_state_last_match = "Ligand 1"
        continue
        
    opt = optimize_threshold(results, min_thresh=5.0, max_thresh=15.0, step=0.5)
    stats = opt['stats']
    last_match_stats = stats
    
    champ_wins = stats.get('lig1_wins', 0)
    challenger_wins = stats.get('lig2_wins', 0)
    
    print(f"Models passing QC: {len(results)}")
    print(f"Champion score: {champ_wins} | Challenger score: {challenger_wins}")
    
    if challenger_wins > champ_wins:
        print(f"Challenger [{challenger_idx}] defeats Champion [{champion_idx}]!")
        champion_idx = challenger_idx
        champion_seq = challenger_seq
        winning_state_last_match = "Ligand 2"
    elif champ_wins > challenger_wins:
        print(f"Champion [{champion_idx}] defends the title!")
        winning_state_last_match = "Ligand 1"
    else:
        # TIE BREAKER
        print("Scores tied! Proceeding to pLDDT tie-breaker...")
        if champ_wins == 0 and challenger_wins == 0:
            print("Neither ligand achieved exclusive binding in any valid models. Champion retains title by default.")
            winning_state_last_match = "Neither"
        else:
            champ_plddt = np.mean([r['mean_plddt'] for r in stats['lig1_results']])
            challenger_plddt = np.mean([r['mean_plddt'] for r in stats['lig2_results']])
            print(f"Champion Avg pLDDT: {champ_plddt:.1f} | Challenger Avg pLDDT: {challenger_plddt:.1f}")
            
            if challenger_plddt > champ_plddt:
                print(f"Challenger [{challenger_idx}] wins by tie-breaker!")
                champion_idx = challenger_idx
                champion_seq = challenger_seq
                winning_state_last_match = "Ligand 2"
            else:
                print(f"Champion [{champion_idx}] wins by tie-breaker!")
                winning_state_last_match = "Ligand 1"
    print("\n")
    
print("=== TOURNAMENT COMPLETE ===")
print(f"ULTIMATE CHAMPION: Candidate [{champion_idx}] ({champion_seq})")


## 3. Visualization of the Final Match
Renders the highest confidence model of the Ultimate Champion winning its last match.

In [ ]:
if not last_match_stats:
    print("No valid models were generated to visualize.")
elif winning_state_last_match == "Neither":
    print("Neither ligand bound in the final match, nothing to visualize.")
else:
    winning_results = []
    
    if winning_state_last_match == "Ligand 1":
        winning_results = last_match_stats.get('lig1_results', [])
    elif winning_state_last_match == "Ligand 2":
        winning_results = last_match_stats.get('lig2_results', [])
        
    if not winning_results:
        print("No structural models available for the winning state to render.")
    else:
        best_model = max(winning_results, key=lambda x: x["mean_plddt"])
        best_pdb = os.path.join(last_match_dir, best_model["pdb_file"])
        
        print(f"Visualizing Final Match. Winning State: {winning_state_last_match}")
        print(f"Best Model: {best_model['pdb_file']} (pLDDT: {best_model['mean_plddt']:.1f})")
        print("Target (Chain A) = Grey | Ligand 1 (Champion) = Blue | Ligand 2 (Challenger) = Red")
        
        if os.path.exists(best_pdb):
            with open(best_pdb, 'r') as f:
                pdb_data = f.read()
                
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            
            # Target (Chain A)
            view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
            # Ligand 1 (Chain B)
            view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'blue'}, 'stick': {'color': 'blue'}})
            # Ligand 2 (Chain C)
            view.setStyle({'chain': 'C'}, {'cartoon': {'color': 'red'}, 'stick': {'color': 'red'}})
            
            view.zoomTo()
            view.show()
        else:
            print(f"Error: Cannot find {best_pdb} to visualize.")
